# ðŸš€ Hunyuan3D-2.1 Cloud Server for Unity Editor Bridge

Run the official **Hunyuan3D-2.1** inference server on a **free Google Colab GPU (T4 / A100 / L4)** and connect it directly to your Unity Editor project.

### ðŸ“Œ How to use:
1. In the Colab menu, go to **Runtime** âž” **Change runtime type** âž” Select **T4 GPU** (free) or better.
2. Run all cells below (or press **Ctrl + F9**).
3. Wait for the final cell to display your **Public Cloudflare Tunnel URL** (e.g. `https://xxxx.trycloudflare.com`).
4. Copy and paste that URL into the **Server URL** field inside Unity (**Tools âž” Hunyuan3D âž” Generator**).
5. Click **Check Server** in Unity âž” Ready to generate 3D models from any Mac, laptop, or low-end PC!

In [ ]:
# Step 1: Check GPU availability
!nvidia-smi

In [ ]:
# Step 2: Clone official Tencent Hunyuan3D-2 repository
import os
if not os.path.exists('Hunyuan3D-2'):
    !git clone https://github.com/Tencent/Hunyuan3D-2.1.git Hunyuan3D-2 || git clone https://github.com/Tencent/Hunyuan3D-2.git Hunyuan3D-2
%cd Hunyuan3D-2

In [ ]:
# Step 3: Install dependencies for shape generation
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install fastapi uvicorn pydantic trimesh rembg
# Install cloudflared to create a free public HTTPS tunnel without account/token
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# Step 4: Apply Unity Bridge optimization patch (step-limit, shape-only, VRAM cleanup)
import os, re

if os.path.exists('api_models.py'):
    with open('api_models.py', 'r') as f: content = f.read()
    content = re.sub(r'le=20', 'le=50', content)
    with open('api_models.py', 'w') as f: f.write(content)
    print('Patched api_models.py (inference steps allowed up to 50)')

if os.path.exists('model_worker.py'):
    with open('model_worker.py', 'r') as f: content = f.read()
    modified = False
    # wrap texture pipeline in try-except for shape-only stability
    if 'except Exception as e:' not in content and 'self.paint_pipeline = Hunyuan3DPaintPipeline(conf)' in content:
        content = content.replace('self.paint_pipeline = Hunyuan3DPaintPipeline(conf)',
            'try:\n            self.paint_pipeline = Hunyuan3DPaintPipeline(conf)\n        except:\n            self.paint_pipeline = None')
        modified = True
    # Inject VRAM cleanup helper after imports
    if '# [Unity Bridge] VRAM cleanup' not in content:
        vram_code = '\n# [Unity Bridge] VRAM cleanup\nimport gc as _gc\ndef _cleanup_vram():\n    _gc.collect()\n    try:\n        import torch\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n    except: pass\n'
        m = re.search(r'^(class\s+\w+)', content, re.MULTILINE)
        if m:
            content = content[:m.start()] + vram_code + '\n' + content[m.start():]
            modified = True
            print('Injected _cleanup_vram() into model_worker.py')
    if modified:
        with open('model_worker.py', 'w') as f: f.write(content)
    print('Patched model_worker.py')

# Inject POST /unload endpoint for on-demand VRAM release
if os.path.exists('api_server.py'):
    with open('api_server.py', 'r') as f: content = f.read()
    if '# [Unity Bridge] VRAM unload' not in content:
        ep = ('\n# [Unity Bridge] VRAM unload endpoint\n'
              '@app.post("/unload")\n'
              'async def unload_vram():\n'
              '    import gc, torch\n'
              '    freed = 0\n'
              '    try:\n'
              '        if torch.cuda.is_available():\n'
              '            before = torch.cuda.memory_allocated()\n'
              '            gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()\n'
              '            freed = (before - torch.cuda.memory_allocated()) / 1024 / 1024\n'
              '    except Exception as e:\n'
              '        return {"status": "error", "message": str(e)}\n'
              '    return {"status": "ok", "freed_mb": round(freed, 1)}\n')
        if 'if __name__' in content:
            content = content.replace('if __name__', ep + '\nif __name__')
        else:
            content += ep
        with open('api_server.py', 'w') as f: f.write(content)
        print('Injected POST /unload endpoint into api_server.py')

In [ ]:
# Step 5: Start Hunyuan3D FastAPI server and Cloudflare Tunnel
import subprocess, time, re

print('Starting local FastAPI server on port 8081...')
server_process = subprocess.Popen([
    'python', 'api_server.py',
    '--host', '127.0.0.1',
    '--port', '8081'
])

# Start Cloudflare Tunnel
print('Starting Cloudflare Tunnel to expose server to Unity...')
tunnel_process = subprocess.Popen([
    'cloudflared', 'tunnel',
    '--url', 'http://127.0.0.1:8081'
], stderr=subprocess.PIPE, universal_newlines=True)

public_url = None
for line in tunnel_process.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print('\n' + '='*60)
print('ðŸŽ‰ HUNYUAN3D CLOUD SERVER IS READY FOR UNITY!')
print('='*60)
print(f'ðŸ‘‰ Copy this Server URL into Unity: {public_url}')
print('='*60 + '\n')

# Keep running
server_process.wait()